# 1. 结构化输出概述
## 1.1 什么是结构化输出

LangChain的结构化输出（Structured Output） 指的是：
> 要求模型最终返回一个符合预定义结构的数据对象，例如固定字段的JSON、Pydantic 模型、TypedDict，而不再是无格式的自然语言文本。

它的核心目标是把“ 自然语言回答 ”变成“ 程序可以稳定消费的数据 ”。
例如，不是让模型输出：

> 盗梦空间在2010年上映，导演是克里斯托弗·诺兰，评分9.3。
而是让它输出成类似这样的结构：

In [1]:
{
    "title": "盗梦空间",
    "year": 2010,
    "director": "克里斯托弗·诺兰",
    "rating": 9.3
}

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 9.3}

LangChain实现结构化输出的方法:

```python
# 一步到位
structured_llm = model.with_structured_output(Person)
person = structured_llm.invoke("张三是一名 30 岁的软件工程师")
# ✅ 自动解析、验证、创建对象
```

1. 传统方式

```python
prompt = "以JSON格式返回：{name, age, occupation}"
response = model.invoke(prompt)
```

2. 手动解析
```python
import json
data = json.loads(response.content)
```
3. 手动验证类型
```python
if not isinstance(data['age'], int):
    raise ValueError("age must be int")
```

4. 手动创建对象
```python
person = Person(**data)
```

## 1.3 结构化输出模式
目前LangChain 1.x 支持多种Schema与结构化输出方式：

* Pydantic（字段校验、描述、嵌套结构，功能最丰富）
* TypedDict（轻量类型约束）
* JSON Schema（与前后端/跨语言接口最通用）
* dataclass

模型对象可以调用 with_structured_output() 绑定输出模式（schema）。

**只有 Pydantic 返回的是Schema类实例，其余三种方式返回的都是 字典 ；也只有 Pydantic 在类型不匹配时会抛出异常。**

问题：现在所有模型都支持“本章要讲解的结构化输出方式”吗？

大部分现代模型支持（通过函数调用）：
1. ✅ OpenAI (gpt-4, gpt-3.5-turbo)
2. ✅ Anthropic (claude-3)
3. ✅ Groq (llama-3)
4. ❌ 某些旧模型不支持

如果模型不支持，调用时会直接报错（比如模型类没有实现工具调用时报 `NotImplementedError`），LangChain **不会**自动回退到"提示词 + JSON 解析"。唯一的自动调整是：ChatOpenAI 遇到不支持 json_schema 的旧模型（gpt-3.5、gpt-4、gpt-4-turbo 等）时，会改用 function_calling 方式并给出警告（见 2.1.2）。

# 2. 四种模式的使用

## 2.1 模式1：Pydantic

它通过在运行时强制执行类型提示，确保数据的正确性和一致性，是 生产场景首选 。

### 2.1.1 基本使用
需要满足的几个要素：
* 所有结构化输出的数据模型都必须继承 BaseModel
* 使用 类型提示 。Pydantic 支持丰富的字段类型：str 、int、float、List[xxx]、Optional[xxx]等
* 使用 Field() 添加字段默认值和描述，帮助 LLM 理解字段含义



In [2]:
import asyncio
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

定义 Pydantic 模型

使用清晰的字段描述；没有描述，LLM 可能格式错误

In [3]:
from pydantic import BaseModel, Field
class Person(BaseModel):
    """人物信息"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

LangChain 会要求 LLM 的输出必须能填充这些字段。

3）使用 with_structured_output 即可引导模型进行结构化输出：

In [4]:
structured_llm = model.with_structured_output(Person)
# 调用
result = structured_llm.invoke("张三是一名 30 岁的软件工程师")

print(f"result:{result}")
print("=================================")
print(type(result))
print("=================================")

result:name='张三' age=30 occupation='软件工程师'
<class '__main__.Person'>


In [5]:
# result 是 Person 实例
print(result.name) # "张三"
print(result.age) # 30
print(result.occupation) # "软件工程师"

张三
30
软件工程师


> 说明：没有描述，LLM 可能格式错误。

In [6]:
from pydantic import BaseModel, Field, SecretStr
class MovieModel(BaseModel):
    """电影的详细信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")
    
model_with_structure = model.with_structured_output(MovieModel)
response = model_with_structure.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

title='盗梦空间' year=2010 director='克里斯托弗·诺兰' rating=8.8
<class '__main__.MovieModel'>


In [7]:
from pydantic import BaseModel, Field
# 定义输出结构
class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0-1之间")
    keywords: list[str] = Field(description="关键词列表")

# ✅ v1.x：使用 with_structured_output
structured_model = model.with_structured_output(SentimentAnalysis)
# 调用
text = "这个课程内容很实用，学到了很多知识，强烈推荐！"
result = structured_model.invoke(
f"分析以下文本的情感：\n{text}"
)
print(f"类型: {type(result)}") # <class 'SentimentAnalysis'>
print(f"情感: {result.sentiment}")
print(f"置信度: {result.confidence}")
print(f"关键词: {result.keywords}")

类型: <class '__main__.SentimentAnalysis'>
情感: positive
置信度: 0.99
关键词: ['实用', '学到了很多知识', '强烈推荐']


### 2.1.2 原理：with_structured_output 做了什么

**先看签名。** 在编辑器里看到的通常是父类 `BaseChatModel` 的接口：

```python
with_structured_output(
  self,
  schema: Union[Dict[str, Any], Type[Any]],
  *,
  include_raw: bool = False,
  **kwargs: Any = {}
) -> Runnable
```

schema: Union 表示参数 schema 接受两种不同类型的输入：字典，或者一个类（`Type`）。

将输出 schema 遵守 Pydantic 模型或 JSON 模式字典。如果是字典，必须包含 'title' 和 'properties' 键。

`ChatOpenAI` 实际运行的是它自己的版本，多了几个参数：

```python
with_structured_output(
    schema=None,
    *,
    method="json_schema",   # 让模型按结构输出的方式，ChatOpenAI 默认 json_schema
    include_raw=False,      # True：同时返回模型的原始消息，解析失败时不抛异常（见 2.1.4）
    strict=None,            # 是否要求输出严格符合 schema
    tools=None,             # json_schema 方式下，可以额外绑定的工具
    **kwargs,
) -> Runnable[LanguageModelInput, dict | BaseModel]
```

**一句话原理：** `with_structured_output` = **把 schema 绑定到模型上**（`bind`）+ **在模型后面接一个解析器**（`| parser`）。源码的最后一行就是：

```python
return llm | output_parser
```

其中 `llm` 是绑定了 schema 的模型，`output_parser` 负责把模型的回复变成 Pydantic 对象。可以用代码拆开看（下面的代码不请求模型）：

In [8]:
structured_llm = model.with_structured_output(Person)

print("返回值类型:", type(structured_llm).__name__)
print("第 1 步:", type(structured_llm.first).__name__, "| 包装的就是原来的 model:", structured_llm.first.bound is model)
print("第 1 步绑定的参数:", list(structured_llm.first.kwargs))
print("response_format =", structured_llm.first.kwargs["response_format"])
print("第 2 步:", type(structured_llm.last).__name__, "| 输出类型:", structured_llm.last.OutputType)

返回值类型: RunnableSequence
第 1 步: _ChatModelBinding | 包装的就是原来的 model: True
第 1 步绑定的参数: ['response_format', 'ls_structured_output_format']
response_format = <class '__main__.Person'>
第 2 步: RunnableBinding | 输出类型: <class '__main__.Person'>


**"让模型按结构输出"有三种方式**，由 `method` 参数决定：

| method | 怎么把 schema 告诉模型 | 能否保证符合 schema | 解析器 |
|---|---|---|---|
| `"json_schema"`（**ChatOpenAI 默认**） | 放进请求的 `response_format`，使用 OpenAI 的 Structured Outputs 功能 | 能（`strict` 模式由服务端保证，见 2.1.4） | 读取 OpenAI SDK 已经解析好的对象 |
| `"function_calling"` | 把 schema 当成一个**工具**，并用 `tool_choice` 强制模型调用它 | `strict=True` 时能；否则基本可靠，但不保证 | `PydanticToolsParser`：读取 tool_call 的参数 |
| `"json_mode"` | `response_format={"type": "json_object"}`，字段要求得自己写在提示词里 | 只保证是合法 JSON，不保证字段 | `PydanticOutputParser`：解析文本 |

笔记里原来写的"使用工具调用生成结构模型"，指的是 `function_calling` 方式。它是父类 `BaseChatModel` 的默认实现，`ChatDeepSeek` 默认也用它；但当前版本的 `ChatOpenAI` 默认用的是 `json_schema`。

`function_calling` 其实就是上一章的知识：`02-tool_define.ipynb` 里提到，Pydantic 类可以直接当工具，"常用于只想让模型按固定结构输出数据的场景"，说的就是这里。

**以 `json_schema` 方式为例，一次调用的完整过程：**

```text
structured_llm.invoke("张三是一名 30 岁的软件工程师")
  │
  ├─ 第 1 步：绑定了 response_format=Person 的模型
  │    ① OpenAI SDK 把 Person 类转换成严格的 JSON Schema（strict: true），随请求发出
  │    ② 服务端按 schema 约束模型生成，返回的 content 是一段 JSON 字符串
  │    ③ SDK 用 Person.model_validate_json(content) 创建 Person 对象，
  │       LangChain 把它放进 AIMessage.additional_kwargs["parsed"]
  │
  └─ 第 2 步：解析器
       ④ 取出 additional_kwargs["parsed"]，作为最终结果返回
```

第 ① 步发出去的 JSON Schema 长这样。下面的代码调用的是 OpenAI SDK 内部的转换函数，只用来观察，平时不需要自己调用：

In [9]:
import json
from openai.lib._parsing._completions import type_to_response_format_param

print(json.dumps(type_to_response_format_param(Person), ensure_ascii=False, indent=2))

{
  "type": "json_schema",
  "json_schema": {
    "schema": {
      "description": "人物信息",
      "properties": {
        "name": {
          "description": "姓名",
          "title": "Name",
          "type": "string"
        },
        "age": {
          "description": "年龄",
          "title": "Age",
          "type": "integer"
        },
        "occupation": {
          "description": "职业",
          "title": "Occupation",
          "type": "string"
        }
      },
      "required": [
        "name",
        "age",
        "occupation"
      ],
      "title": "Person",
      "type": "object",
      "additionalProperties": false
    },
    "name": "Person",
    "strict": true
  }
}


和上一章工具说明里的 `parameters` 很像，多了两处：

- `"strict": true`：要求服务端严格按 schema 生成
- `"additionalProperties": false`：不允许出现 schema 里没有的字段

第 ②、③ 步可以用 `include_raw=True` 观察：它会把模型的原始消息 `raw` 和解析结果 `parsed` 一起返回。下面这次会真正请求模型：

In [10]:
structured_llm_raw = model.with_structured_output(Person, include_raw=True)
output = structured_llm_raw.invoke("张三是一名 30 岁的软件工程师")

print("返回值的键:", list(output))
print()
print("raw.content（模型实际返回的文本）:", repr(output["raw"].content))
print("raw.additional_kwargs['parsed']:", repr(output["raw"].additional_kwargs.get("parsed")))
print("finish_reason:", output["raw"].response_metadata.get("finish_reason"))
print()
print("parsed:", repr(output["parsed"]))
print("parsing_error:", output["parsing_error"])

返回值的键: ['raw', 'parsed', 'parsing_error']

raw.content（模型实际返回的文本）: '{"age":30,"name":"张三","occupation":"软件工程师"}'
raw.additional_kwargs['parsed']: Person(name='张三', age=30, occupation='软件工程师')
finish_reason: stop

parsed: Person(name='张三', age=30, occupation='软件工程师')
parsing_error: None


可以看到：模型返回的 `content` 只是一段 **JSON 字符串**；`Person` 对象是 SDK 解析出来、放在 `additional_kwargs["parsed"]` 里的，解析器再把它取出来作为 `parsed`。字段顺序和 `Person` 里定义的不一样也没关系，解析 JSON 时是按键名取值的。

### 2.1.3 返回值 Runnable 是什么意思

签名最后的 `-> Runnable[LanguageModelInput, dict | BaseModel]` 说明：`with_structured_output` 返回的**不是模型本身**，而是一个新的 Runnable。

**Runnable 是什么？** 它是 LangChain 里"可以运行的组件"的统一接口。模型、提示词模板、输出解析器、工具都是 Runnable，它们都有同一套方法：

| 方法 | 作用 |
|---|---|
| `invoke(输入)` | 处理一个输入，返回一个输出 |
| `batch([输入1, 输入2])` | 批量处理多个输入 |
| `stream(输入)` | 流式返回输出 |
| `ainvoke` / `abatch` / `astream` | 上面三个方法的异步版本 |
| `a \| b` | 把两个 Runnable 串起来：`a` 的输出作为 `b` 的输入，得到一个新的 Runnable（`RunnableSequence`） |

所以只要是 Runnable，调用方式都一样，也都能用 `|` 拼在一起。

**方括号里是什么？** `Runnable[输入类型, 输出类型]`：

- `LanguageModelInput`：输入和普通模型一样，可以是字符串、消息列表或提示词模板生成的 PromptValue
- `dict | BaseModel`：输出是字典或 Pydantic 对象（取决于 schema 的类型），**不再是 `AIMessage`**

**具体是哪个 Runnable？** 2.1.2 已经看到，是一个两步的 `RunnableSequence`：绑定了 schema 的模型 → 解析器。它和 `提示词 | 模型 | 解析器` 这样的链是同一种东西，只是 LangChain 帮你拼好了。

和上一章的 `bind_tools` 对比：

| | `bind_tools` | `with_structured_output` |
|---|---|---|
| 做了什么 | 翻译工具 + `bind` | `bind` schema + 接上解析器 |
| 返回的 Runnable | `RunnableBinding` | `RunnableSequence`（第一步就是一个 `RunnableBinding`） |
| 输出 | `AIMessage`（工具调用在 `tool_calls` 里） | Pydantic 对象或字典 |
| 原来的 `model` | 不变 | 不变 |

因为返回的是 Runnable，它可以像普通模型一样接在提示词模板后面（下面的代码不请求模型）：

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "从用户的描述中提取人物信息"),
    ("human", "{text}"),
])

chain = prompt | structured_llm

print("链的每一步:", [type(step).__name__ for step in chain.steps])
print("链的输出类型:", chain.OutputType)
# chain.invoke({"text": "李四今年 25 岁，是一名医生"}) 会返回一个 Person 对象

链的每一步: ['ChatPromptTemplate', '_ChatModelBinding', 'RunnableBinding']
链的输出类型: <class '__main__.Person'>


`prompt | structured_llm` 得到的链有三步：提示词模板 → 绑定了 schema 的模型 → 解析器。LangChain 把两个序列"拍平"成了一个，整条链的输出类型仍然是 `Person`。

### 2.1.4 为什么模型"一定"会返回 Pydantic 对象？

先纠正两个说法：

1. **拿到的是 Pydantic 类的实例（对象），不是类本身。** `type(result)` 是类 `Person`，`result` 是用这个类创建出来的一个对象。
2. **Pydantic 对象不是模型给的。** 模型的接口只能返回文本，2.1.2 里的 `raw.content` 就是一段 JSON 字符串。Pydantic 对象是在**你自己的电脑上**，由 OpenAI SDK 调用 `Person.model_validate_json(文本)` 创建的。你传进去的是 `Person` 类，创建出来的自然就是 `Person` 对象。

    这也解释了 1.3 的结论：schema 是 TypedDict 或 JSON Schema 时，没有可以实例化的 Pydantic 类，解析器只能把 JSON 解析成字典。

所以问题其实是：**为什么模型返回的 JSON 文本，总能被成功解析成 `Person`？** 靠的是三层保障：

| 层 | 谁负责 | 做了什么 |
|---|---|---|
| ① 告诉模型要什么 | LangChain + OpenAI SDK | 把 `Person` 转成 JSON Schema 随请求发出：字段名、类型、描述、哪些必填 |
| ② 限制模型能输出什么 | OpenAI 服务端 | `strict: true` 时启用**约束解码**：模型每生成一个 token，只能从"能让输出继续符合 schema"的 token 里选。比如 `age` 的位置只能生成数字 |
| ③ 检查输出对不对 | Pydantic（在你的电脑上） | `model_validate_json` 逐个字段校验类型，不合格就抛出 `ValidationError` |

关键是第 ② 层。只靠提示词要求"按格式输出"时，模型仍然可能写错；约束解码则是在生成时就**直接禁止**不合规的输出。所以 OpenAI 说，开启 strict 后输出"保证"符合 schema。

第 ③ 层做的事可以直接演示。假如没有第 ② 层，模型把年龄写成了 `"三十岁"`：

In [12]:
from pydantic import ValidationError

# 模型按 schema 生成的 JSON：解析成功
good_json = '{"name": "张三", "age": 30, "occupation": "软件工程师"}'
print(repr(Person.model_validate_json(good_json)))

# 假如模型没按 schema 生成：age 不是整数，解析失败
bad_json = '{"name": "张三", "age": "三十岁", "occupation": "软件工程师"}'
try:
    Person.model_validate_json(bad_json)
except ValidationError as e:
    print("ValidationError ->", " ".join(line.strip() for line in str(e).splitlines()[1:3]))

Person(name='张三', age=30, occupation='软件工程师')
ValidationError -> age Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='三十岁', input_type=str]


**所以并不是"一定"。** 下面这些情况仍然拿不到 Pydantic 对象：

| 情况 | 结果 |
|---|---|
| 模型出于安全原因拒绝回答 | 抛出 `OpenAIRefusalError` |
| 输出太长被截断（达到 `max_tokens`），JSON 不完整 | 抛出 `openai.LengthFinishReasonError` |
| 服务端不支持或没有执行 strict，比如部分中转服务、兼容 OpenAI 接口的其他厂商 | 第 ② 层失效，可能抛出 `ValidationError` |
| 使用 `method="json_mode"` | 只保证是合法 JSON，字段可能不对，可能抛出 `ValidationError` |
| Pydantic 里写了自定义校验规则（如 `@field_validator`） | 模型看不到这些规则，可能抛出 `ValidationError` |

**用 `include_raw=True` 处理失败。** 2.1.2 已经用过它：返回值变成字典 `{"raw": 原始消息, "parsed": 解析结果, "parsing_error": 解析异常}`。解析失败时不抛异常，而是 `parsed=None`，异常记在 `parsing_error` 里。

但它只能兜住**解析器那一步**的错误，而且两种 method 的解析位置不同：

- `method="json_schema"`（ChatOpenAI 默认）：JSON → `Person` 这一步是 OpenAI SDK 在**请求模型的过程中**完成的。所以 JSON 不符合 schema 或输出被截断时，异常直接从模型那一步抛出，`include_raw=True` 兜不住。它能兜住的是"模型拒绝回答"（`parsing_error` 为 `OpenAIRefusalError`）。
- `method="function_calling"`：模型只返回 tool_call，把参数转成 `Person` 是解析器做的。所以 schema 校验失败时，`include_raw=True` 能兜住（`parsing_error` 为 `ValidationError`）。

### 2.1.5 小结

| 问题 | 答案 |
|---|---|
| `with_structured_output` 做了什么？ | `bind`（把 schema 绑定到模型上）+ `\| parser`（在模型后面接一个解析器）。ChatOpenAI 默认用 `json_schema` 方式，把 Pydantic 类转成严格的 JSON Schema，放进请求的 `response_format` |
| 返回值 Runnable 是什么？ | 一个两步的 `RunnableSequence`：绑定了 schema 的模型 → 解析器。它和模型一样可以 `invoke` / `batch` / `stream`，可以用 `\|` 串联，但输出从 `AIMessage` 变成了 Pydantic 对象 |
| 为什么总能拿到 Pydantic 对象？ | 模型只返回 JSON 文本；strict 模式的约束解码保证文本符合 schema；SDK 再用 `Person.model_validate_json` 把文本变成 `Person` 对象。拒绝回答、输出被截断、服务端不支持 strict 时，仍然会失败 |